In [13]:
import pandas as pd
import re

# 1. Load the dataset
df = pd.read_csv('IMDB Dataset.csv')

# 2. Basic Preprocessing
df['sentiment'] = df['sentiment'].str.strip().str.lower()

# 3. Define Keywords
pos_keywords = ['excellent', 'favorite']
neg_keywords = ['worst', 'waste']
all_keywords = pos_keywords + neg_keywords

# 4 verification of the work.
print(f"Total reviews: {len(df)}")
print(f"Unique sentiment values: {df['sentiment'].unique()}")
print(f"Keywords initialized: {all_keywords}")

df.head()

Total reviews: 50000
Unique sentiment values: ['positive' 'negative']
Keywords initialized: ['excellent', 'favorite', 'worst', 'waste']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [14]:
total_count = len(df)
pos_reviews = df[df['sentiment'] == 'positive']
neg_reviews = df[df['sentiment'] == 'negative']

p_positive = len(pos_reviews) / total_count
p_negative = len(neg_reviews) / total_count

print(f"Prior P(Positive): {p_positive:.4f}")

Prior P(Positive): 0.5000


In [16]:
def compute_bayes(keyword, pos_df, neg_df, p_pos, p_neg):
    # Regex \b ensures we match the whole word only (not "wasteful" for "waste")
    pattern = r'\b' + keyword + r'\b'

    # Likelihoods: P(keyword | Sentiment)
    # We use .mean() on a boolean series to get the probability directly
    p_k_given_pos = pos_df['review'].str.contains(pattern, case=False, regex=True).mean()
    p_k_given_neg = neg_df['review'].str.contains(pattern, case=False, regex=True).mean()

    # Marginal: P(keyword) = P(K|Pos)P(Pos) + P(K|Neg)P(Neg)
    p_k = (p_k_given_pos * p_pos) + (p_k_given_neg * p_neg)

    # Posterior: P(Positive | keyword) = [P(K|Pos) * P(Pos)] / P(K)
    posterior = (p_k_given_pos * p_pos) / p_k if p_k > 0 else 0

    return {
        'Keyword': keyword,
        'Prior': round(p_pos, 4),
        'Likelihood': round(p_k_given_pos, 4),
        'Marginal': round(p_k, 4),
        'Posterior': round(posterior, 4)
    }

# Run the computation for all keywords
results = [compute_bayes(kw, pos_reviews, neg_reviews, p_positive, p_negative) for kw in all_keywords]

In [17]:
# Create a summary table
results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

# Quick Interpretation check
for r in results:
    status = "Positive Signal" if r['Posterior'] > 0.5 else "Negative Signal"
    print(f"Keyword '{r['Keyword']}': Posterior is {r['Posterior']} -> {status}")

| Keyword   |   Prior |   Likelihood |   Marginal |   Posterior |
|:----------|--------:|-------------:|-----------:|------------:|
| excellent |     0.5 |       0.1147 |     0.071  |      0.8074 |
| favorite  |     0.5 |       0.0641 |     0.0425 |      0.7546 |
| worst     |     0.5 |       0.0164 |     0.0887 |      0.0927 |
| waste     |     0.5 |       0.007  |     0.0507 |      0.0691 |
Keyword 'excellent': Posterior is 0.8074 -> Positive Signal
Keyword 'favorite': Posterior is 0.7546 -> Positive Signal
Keyword 'worst': Posterior is 0.0927 -> Negative Signal
Keyword 'waste': Posterior is 0.0691 -> Negative Signal
